# 🚕 taxSEE
## A _fare_-ly good predictor to see how much you'll pay

This notebook loads an **existing trained Random Forest model** from MLflow and makes fare predictions using **9 user-friendly features** — no retraining required.

Just provide trip details like distance, duration, passenger count, and pickup time, and the model returns a predicted total fare.

| Metric | Value |
|--------|-------|
| **MAE** | ~$1.60 |
| **R²** | 0.887 |

> The model was trained on ~46 million NYC taxi trips. See `model-assessment-report.md` for the full evaluation.

In [0]:
import os
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/students_data/team3-taxi/mlflow_tmp"

import mlflow.spark
from pyspark.sql import functions as F
from pyspark.sql.functions import col

## Step 1: Load the Trained Model from MLflow

In [0]:
RUN_ID = "09418df8decf4fe8af5e05732331edff"
model = mlflow.spark.load_model(f"runs:/{RUN_ID}/taxi_fare_model")
print(f"✅ Model loaded from MLflow run: {RUN_ID}")
print("   Pipeline includes: VectorAssembler + StandardScaler + RandomForest")

## Step 2: Feature Reference

The model loaded above is a **Spark ML PipelineModel** that bundles the `VectorAssembler`, `StandardScaler`, and `RandomForestRegressionModel` into a single object.

No separate scaler re-fitting is needed — just pass raw feature columns and the pipeline handles everything.

In [0]:
# Feature columns (must match the pipeline's expected input order)
FEATURE_COLS = [
    'trip_distance', 'trip_duration_minutes', 'passenger_count',
    'vendor_key', 'rate_code_key', 'payment_type_key',
    'pickup_hour', 'pickup_day_of_week', 'pickup_is_weekend',
]

print(f"✅ {len(FEATURE_COLS)} features defined: {FEATURE_COLS}")

## Step 3: Predict Function

The helper function below takes simple, human-readable inputs, creates a one-row DataFrame, and passes it straight through `model.transform()`. The PipelineModel handles feature assembly and scaling internally — no manual preprocessing needed.

In [0]:
def predict_fare(trip_distance, trip_duration_minutes, passenger_count=1,
                 pickup_hour=12, pickup_day_of_week=3, pickup_is_weekend=0,
                 vendor_key=1, rate_code_key=1, payment_type_key=1):
    """
    Predict NYC taxi fare from user-friendly inputs.
    The PipelineModel handles feature assembly and scaling internally.

    Args:
        trip_distance:        Distance in miles (e.g. 3.5)
        trip_duration_minutes: Trip time in minutes (e.g. 15)
        passenger_count:      Passengers (1–6, default 1)
        pickup_hour:          Hour of pickup 0–23 (default 12)
        pickup_day_of_week:   1=Sun 2=Mon 3=Tue 4=Wed 5=Thu 6=Fri 7=Sat (default 3)
        pickup_is_weekend:    0=weekday 1=weekend (default 0)
        vendor_key:           Taxi company 1 or 2 (default 1)
        rate_code_key:        1=Standard 2=JFK 3=Newark 4=Nassau 5=Negotiated (default 1)
        payment_type_key:     1=Card 2=Cash 3=No charge 4=Dispute (default 1)

    Returns:
        Predicted total fare in USD
    """
    trip = {
        'trip_distance':         float(trip_distance),
        'trip_duration_minutes': float(trip_duration_minutes),
        'passenger_count':       float(passenger_count),
        'vendor_key':            float(vendor_key),
        'rate_code_key':         float(rate_code_key),
        'payment_type_key':      float(payment_type_key),
        'pickup_hour':           float(pickup_hour),
        'pickup_day_of_week':    float(pickup_day_of_week),
        'pickup_is_weekend':     float(pickup_is_weekend),
    }
    trip_df = spark.createDataFrame([trip])
    return round(model.transform(trip_df).select("prediction").first()[0], 2)

print("✅ predict_fare() is ready to use")

## Step 4: Example Predictions

Below are several realistic trip scenarios. The 9 input features are:

| Feature | Description | Typical Values |
|---------|-------------|----------------|
| `trip_distance` | Distance in miles | 0.5 – 30 |
| `trip_duration_minutes` | Duration in minutes | 2 – 60 |
| `passenger_count` | Number of passengers | 1 – 6 |
| `pickup_hour` | Hour of day (24h) | 0 – 23 |
| `pickup_day_of_week` | Day of week | 1=Sun … 7=Sat |
| `pickup_is_weekend` | Weekend flag | 0 or 1 |
| `vendor_key` | Taxi company | 1 or 2 |
| `rate_code_key` | Rate code | 1=Std, 2=JFK, 3=Newark |
| `payment_type_key` | Payment method | 1=Card, 2=Cash |

In [0]:
scenarios = [
    ("Short weekday morning (1 mi, 5 min, 8 am Tue)",
     dict(trip_distance=1.0, trip_duration_minutes=5, pickup_hour=8, pickup_day_of_week=3)),
    ("Medium weekday afternoon (5 mi, 20 min, 2 pm Wed)",
     dict(trip_distance=5.0, trip_duration_minutes=20, passenger_count=2, pickup_hour=14, pickup_day_of_week=4)),
    ("Long weekend evening (15 mi, 45 min, 9 pm Sat)",
     dict(trip_distance=15.0, trip_duration_minutes=45, passenger_count=3, pickup_hour=21, pickup_day_of_week=7, pickup_is_weekend=1)),
    ("JFK flat-rate (18 mi, 50 min, 6 am Mon)",
     dict(trip_distance=18.0, trip_duration_minutes=50, pickup_hour=6, pickup_day_of_week=2, rate_code_key=2)),
    ("Late-night weekend hop (2 mi, 10 min, 1 am Sun)",
     dict(trip_distance=2.0, trip_duration_minutes=10, passenger_count=2, pickup_hour=1, pickup_day_of_week=1, pickup_is_weekend=1)),
]

print("🚕 Example Fare Predictions")
print("=" * 60)
for desc, args in scenarios:
    fare = predict_fare(**args)
    print(f"\n  {desc}")
    print(f"  → Predicted fare: ${fare:.2f}")
print("\n" + "=" * 60)

## Step 5: Try Your Own Trip!

Edit the values in the cell below to get a fare prediction for any trip you like. Just change the numbers and re-run the cell.

In [0]:
# ── Edit these values to predict your own trip fare ──
my_fare = predict_fare(
    trip_distance         = 3.5,   # miles
    trip_duration_minutes = 15,    # minutes
    passenger_count       = 1,     # 1–6
    pickup_hour           = 17,    # 0–23 (17 = 5 pm)
    pickup_day_of_week    = 6,     # 1=Sun … 7=Sat (6 = Fri)
    pickup_is_weekend     = 0,     # 0 = weekday, 1 = weekend
    vendor_key            = 1,     # 1 or 2
    rate_code_key         = 1,     # 1=Standard, 2=JFK, 3=Newark
    payment_type_key      = 1,     # 1=Card, 2=Cash
)

print(f"💰 Predicted fare: ${my_fare:.2f}")

## Step 6: Batch Prediction on Real Trips

To validate the model, we score 10 randomly sampled trips from the actual dataset and compare predicted fares against the real `total_amount`.

In [0]:
from pyspark.sql.functions import abs as spark_abs, rand

# Load trip data joined with time features for batch validation
trips = spark.table("students_data.`team3-taxi`.fact_trip")
dim_hour = (
    spark.table("students_data.`team3-taxi`.dim_datetime_hour")
    .select(
        col("datetime_hour_key"),
        col("hour").alias("pickup_hour"),
        col("day_of_week").alias("pickup_day_of_week"),
        col("is_weekend").alias("pickup_is_weekend"),
    )
)
data = (
    trips.join(dim_hour, trips.pickup_datetime_hour_key == dim_hour.datetime_hour_key, "left")
    .withColumn("pickup_is_weekend", col("pickup_is_weekend").cast("int"))
    .select(FEATURE_COLS + ["total_amount"])
    .dropna()
)

# Sample 10 real valid trips
sample = (
    data
    .filter(col("trip_distance") >= 1)
    .filter(col("trip_duration_minutes") > 0)
    .filter(col("total_amount") > 0)
    .orderBy(rand(seed=124))
    .limit(10)
)

# Pipeline handles assembling and scaling internally
scored = model.transform(sample)

results = scored.select(
    "trip_distance", "trip_duration_minutes", "passenger_count",
    "pickup_hour", "pickup_day_of_week",
    F.round("total_amount", 2).alias("actual_fare"),
    F.round("prediction", 2).alias("predicted_fare"),
    F.round(spark_abs(col("prediction") - col("total_amount")), 2).alias("error"),
).orderBy("trip_distance")

print("📊 Model vs Actual Fares (10 Random Trips)")
display(results)

---
### Model Details

| Property | Value |
|----------|-------|
| **Algorithm** | Random Forest (100 trees) via Spark ML Pipeline |
| **Training data** | \~46 million NYC taxi trips |
| **Features** | 9 user-friendly inputs |
| **MAE** | \~$1.60 |
| **R²** | 0.889 |
| **MLflow Run ID** | `09418df8decf4fe8af5e05732331edff` |
| **Experiment path** | `/Users/neil.obriain@kainos.com/taxi-fare-prediction` |
| **Training notebook** | `task4-refine` (cell 40) |

The model is a full `PipelineModel` (VectorAssembler → StandardScaler → RandomForest). No separate scaler setup is needed — just call `model.transform(raw_df)`.

For full evaluation details including residual analysis and feature importance, see **`model-assessment-report.md`** in this folder.